In [2]:
import sys
sys.path.append("../")

from src.loaders.document_loader import EnterpriseDocumentLoader
from src.splitters.chunker import EnterpriseChunker
from src.embeddings.embedding_manager import EmbeddingManager
from vector_store.faiss_store import EnterpriseFAISS

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever


In [3]:
loader = EnterpriseDocumentLoader()

documents = loader.load_directory("../datasets/military")

chunker = EnterpriseChunker(
    chunk_size=500,
    overlap=100
)

chunks = chunker.split_documents(documents)

print(len(chunks))

29


In [4]:
embedding_model = EmbeddingManager(
    provider="ollama",
    model_name="nomic-embed-text"
)

faiss_store = EnterpriseFAISS(
    embedding_model
)

faiss_store.create(chunks)

vector_retriever = faiss_store.retriever(k=5)

In [5]:
bm25 = BM25Retriever.from_documents(chunks)

bm25.k = 5

In [6]:
docs = bm25.invoke(
    "Radar Maintenance"
)

len(docs)

5

In [7]:
bm_docs = bm25.invoke(
    "Radar Calibration"
)

for doc in bm_docs:

    print(doc.metadata)

    print(doc.page_content[:200])

    print()

{'source': '..\\datasets\\military\\Weapons.csv', 'row': 14, 'filename': 'Weapons.csv', 'extension': '.csv', 'domain': 'military', 'chunk_id': 28, 'chunk_size': 243}
Weapon_ID: WPN-015
Weapon_Name: EBR-14
Category: Marksman Rifle
Caliber: 7.62x51mm
Fire_Mode: Semi-Auto
Mag_Size: 10
Effective_Range_Meters: 700
Attachment_Slots: 5
Base_Damage: 55
Recoil_Rating: Medi

{'source': '..\\datasets\\military\\Weapons.csv', 'row': 13, 'filename': 'Weapons.csv', 'extension': '.csv', 'domain': 'military', 'chunk_id': 27, 'chunk_size': 232}
Weapon_ID: WPN-014
Weapon_Name: P90
Category: SMG
Caliber: 5.7x28mm
Fire_Mode: Full-Auto/Semi
Mag_Size: 50
Effective_Range_Meters: 275
Attachment_Slots: 5
Base_Damage: 28
Recoil_Rating: Low-Medium
Unl

{'source': '..\\datasets\\military\\Weapons.csv', 'row': 12, 'filename': 'Weapons.csv', 'extension': '.csv', 'domain': 'military', 'chunk_id': 26, 'chunk_size': 234}
Weapon_ID: WPN-013
Weapon_Name: .357 Magnum
Category: Pistol
Caliber: .357 Magnum
Fire_Mode: Semi-

In [8]:
faiss_docs = vector_retriever.invoke(
    "Radar Calibration"
)

for doc in faiss_docs:

    print(doc.metadata)

    print(doc.page_content[:200])

    print()

{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 13, 'chunk_size': 226}
===
           REMEMBER:  RADAR  DISCIPLINE  SAVES  LIVES.  STAY  VIGILANT.  ==================================================

{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 12, 'chunk_size': 207}
in  known  enemy  territory,  then  reposition     to  ambush  counter-UAV  teams  -  Overwatch  Rotation:  Coordinate  with  squad  to  maintain  100%  radar     uptime  through  sequential  UAV  dep

{'producer': 

In [9]:
hybrid = EnsembleRetriever(

    retrievers=[
        bm25,
        vector_retriever
    ],

    weights=[
        0.4,
        0.6
    ]
)

In [10]:
docs = hybrid.invoke(
    "Radar Calibration Procedure"
)

len(docs)

10

In [11]:
for i, doc in enumerate(docs):

    print("="*70)

    print(i+1)

    print(doc.metadata)

    print()

    print(doc.page_content[:300])

1
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 12, 'chunk_size': 207}

in  known  enemy  territory,  then  reposition     to  ambush  counter-UAV  teams  -  Overwatch  Rotation:  Coordinate  with  squad  to  maintain  100%  radar     uptime  through  sequential  UAV  deployment
2
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 13, 'chunk_size': 226}

===
           REMEMBER:  RADAR  DISCIPLINE  SAVES  LIVES.  STAY  VIGILANT.  =====================================================

In [13]:
from src.retriever.hybrid_retriever import EnterpriseHybridRetriever

retriever = EnterpriseHybridRetriever(

    chunks,

    vector_retriever,

    bm25_weight=0.4,

    vector_weight=0.6,

    k=5

)

docs = retriever.search(
    "Surface surveillance radar"
)

print(len(docs))

7


In [14]:
queries = [

    "Missile launch procedure",

    "Radar calibration",

    "Tank maintenance",

    "Emergency shutdown"

]

for q in queries:

    print("="*70)

    print("Query :", q)

    docs = retriever.search(q)

    print("Retrieved :", len(docs))

Query : Missile launch procedure
Retrieved : 9
Query : Radar calibration
Retrieved : 10
Query : Tank maintenance
Retrieved : 10
Query : Emergency shutdown
Retrieved : 10


In [15]:
queries = [

    "anticipatory bail",

    "contract breach",

    "criminal appeal",

    "consumer protection"

]

for q in queries:

    docs = retriever.search(q)

    print(q, len(docs))

anticipatory bail 10
contract breach 10
criminal appeal 9
consumer protection 9
